# Fase 5 · Wilcoxon Top-3: comparativa estadística

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Email** | mjmorteruiz@uoc.edu (UOC) / morte@uji.es (UJI) |
| **Fase** | Fase 5 — Modelado |
| **Tipo** | Análisis estadístico complementario |

---

## 🎯 Qué hace

Aplica el **test de Wilcoxon signed-rank pareado** entre los **3 mejores modelos**
del ranking F5:

- **XGBoost (`none`)** — 1ª posición ranking F5 — AUC=0.9537
- **LightGBM (`none`)** — 4ª posición ranking F5 — AUC=0.9521
- **Stacking (`balanced`)** — 7ª posición ranking F5 — AUC=0.9509

Realiza las **3 comparaciones pareadas** y devuelve una tabla resumen con
p-values y veredictos para cada par.

## 📋 Requisitos

- `data/05_modelado/X_train.parquet`, `y_train.parquet` (m01a)
- `data/05_modelado/models/XGBoost__none.pkl` (m03)
- `data/05_modelado/models/LightGBM__none.pkl` (m03)
- `data/05_modelado/models/Stacking__balanced.pkl` (m06)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/05_modelado/results/wilcoxon_top3.json` | Resultados completos (folds + tests) |

## 🔄 Flujo

```
X_train, y_train (raw)
    ↓ 5-Fold StratifiedKFold (random_state=42, mismos folds para los 3 modelos)
    ↓ Por cada fold:
        ├─ XGBoost.fit  → AUC, F1
        ├─ LightGBM.fit → AUC, F1
        └─ Stacking.fit → AUC, F1
    ↓ 3 tests Wilcoxon pareados:
        ├─ XGBoost vs LightGBM
        ├─ XGBoost vs Stacking
        └─ LightGBM vs Stacking
→ wilcoxon_top3.json + tabla resumen
```

## ⚠️ Decisiones metodológicas

1. **CV pareado** — mismos folds para los 3 modelos
2. **`random_state=42`** — reproducibilidad total
3. **`alternative='two-sided'`** — no asumimos previamente cuál gana
4. **α = 0.05** — nivel de significación estándar
5. **Stacking es lento** — ~9-12 min total (3-4 min × Stacking × 5 folds)

## 📚 Interpretación

Para cada par:
- `p < 0.05` → diferencia significativa
- `p ≥ 0.05` → empate técnico → elegir por OTRO criterio

## ➡️ Siguiente

Documentar la tabla en la memoria del TFM (Anexo D).


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN Y CARGA DE DATOS
# ============================================================================

import sys, json, warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

# ── ROOT robusto ──────────────────────────────────────────────────────────────
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# ── Imports del proyecto ──────────────────────────────────────────────────────
from src.utils import formato_numero_es
fmt = formato_numero_es

# ── Rutas ─────────────────────────────────────────────────────────────────────
RUTA_MODELADO = ROOT / 'data' / '05_modelado'
RUTA_MODELS   = RUTA_MODELADO / 'models'
RUTA_RESULTS  = RUTA_MODELADO / 'results'

# ── Cargar datos originales ───────────────────────────────────────────────────────
X_train = pd.read_parquet(RUTA_MODELADO / 'X_train.parquet')
y_train = pd.read_parquet(RUTA_MODELADO / 'y_train.parquet').squeeze()

print('═' * 60)
print('Datos cargados')
print('═' * 60)
print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape} | abandono: {(y_train==1).mean()*100:.1f}%')


════════════════════════════════════════════════════════════
Datos cargados
════════════════════════════════════════════════════════════
X_train: (26896, 27)
y_train: (26896,) | abandono: 29.2%


In [2]:
# ============================================================================
# CELDA 2: CARGAR LOS 3 MODELOS A COMPARAR
# ============================================================================
# Los .pkl se cargan como REFERENCIA de configuración. En el siguiente paso
# se clonan y reentrenan en cada fold del CV pareado para garantizar que
# el test de Wilcoxon sea metodológicamente correcto.
# ============================================================================

modelo_xgb = joblib.load(RUTA_MODELS / 'XGBoost__none.pkl')
modelo_lgb = joblib.load(RUTA_MODELS / 'LightGBM__none.pkl')
modelo_stk = joblib.load(RUTA_MODELS / 'Stacking__balanced.pkl')

print('Modelos cargados (referencia de configuración):')
print(f'  ✅ XGBoost__none.pkl       → tipo: {type(modelo_xgb).__name__}')
print(f'  ✅ LightGBM__none.pkl      → tipo: {type(modelo_lgb).__name__}')
print(f'  ✅ Stacking__balanced.pkl  → tipo: {type(modelo_stk).__name__}')

print()
print('⚠️  Stacking es lento (~3-4 min por fold). Tiempo total estimado: 9-12 min.')


Modelos cargados (referencia de configuración):
  ✅ XGBoost__none.pkl       → tipo: Pipeline
  ✅ LightGBM__none.pkl      → tipo: Pipeline
  ✅ Stacking__balanced.pkl  → tipo: Pipeline

⚠️  Stacking es lento (~3-4 min por fold). Tiempo total estimado: 9-12 min.


In [3]:
# ============================================================================
# CELDA 3: 5-FOLD CV PAREADO — mismos folds para los 3 modelos
# ============================================================================
# Estrategia: usamos StratifiedKFold con random_state=42 (igual que F5) para
# garantizar que los 3 modelos se entrenan/evalúan EN EL MISMO SPLIT. Esto
# elimina la varianza por partición y permite Wilcoxon pareado correcto.
# ============================================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, f1_score
import time

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

aucs = {'XGBoost': [], 'LightGBM': [], 'Stacking': []}
f1s  = {'XGBoost': [], 'LightGBM': [], 'Stacking': []}

modelos_ref = {
    'XGBoost':  modelo_xgb,
    'LightGBM': modelo_lgb,
    'Stacking': modelo_stk,
}

print('5-Fold Stratified CV pareado (random_state=42)')
print('═' * 70)

t0_total = time.time()

for fold, (idx_tr, idx_va) in enumerate(cv.split(X_train, y_train), 1):
    X_tr = X_train.iloc[idx_tr]
    X_va = X_train.iloc[idx_va]
    y_tr = y_train.iloc[idx_tr]
    y_va = y_train.iloc[idx_va]
    
    print(f'\n  Fold {fold}:')
    
    for nombre, modelo_ref in modelos_ref.items():
        t0 = time.time()
        m = clone(modelo_ref)
        m.fit(X_tr, y_tr)
        p = m.predict_proba(X_va)[:, 1]
        pred = (p >= 0.5).astype(int)
        auc = roc_auc_score(y_va, p)
        f1_val = f1_score(y_va, pred)
        t = time.time() - t0
        
        aucs[nombre].append(auc)
        f1s[nombre].append(f1_val)
        
        print(f'    {nombre:10s}: AUC={auc:.4f}  F1={f1_val:.4f}  ({t:.1f}s)')

t_total = time.time() - t0_total
print(f'\n═' * 70)
print(f'\n⏱️  Tiempo total: {t_total/60:.1f} min')
print(f'\n📊 Resumen (mean ± std):')
for nombre in modelos_ref:
    print(f'  {nombre:10s}: AUC = {np.mean(aucs[nombre]):.4f} ± {np.std(aucs[nombre]):.4f}  '
          f'|  F1 = {np.mean(f1s[nombre]):.4f} ± {np.std(f1s[nombre]):.4f}')


5-Fold Stratified CV pareado (random_state=42)
══════════════════════════════════════════════════════════════════════

  Fold 1:
    XGBoost   : AUC=0.9536  F1=0.8325  (1.5s)
    LightGBM  : AUC=0.9520  F1=0.8306  (1.7s)
    Stacking  : AUC=0.9504  F1=0.8387  (341.0s)

  Fold 2:
    XGBoost   : AUC=0.9530  F1=0.8343  (0.8s)
    LightGBM  : AUC=0.9516  F1=0.8306  (1.5s)
    Stacking  : AUC=0.9495  F1=0.8259  (316.0s)

  Fold 3:
    XGBoost   : AUC=0.9560  F1=0.8258  (1.8s)
    LightGBM  : AUC=0.9541  F1=0.8210  (2.1s)
    Stacking  : AUC=0.9545  F1=0.8260  (323.1s)

  Fold 4:
    XGBoost   : AUC=0.9508  F1=0.8281  (1.6s)
    LightGBM  : AUC=0.9500  F1=0.8285  (3.1s)
    Stacking  : AUC=0.9486  F1=0.8252  (319.5s)

  Fold 5:
    XGBoost   : AUC=0.9539  F1=0.8290  (1.3s)
    LightGBM  : AUC=0.9526  F1=0.8282  (1.6s)
    Stacking  : AUC=0.9525  F1=0.8277  (494.9s)

═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═


In [4]:
# ============================================================================
# CELDA 4: TEST DE WILCOXON SIGNED-RANK PAREADO — 3 COMPARACIONES
# ============================================================================
# 3 comparaciones pareadas:
#   • XGBoost  vs LightGBM
#   • XGBoost  vs Stacking
#   • LightGBM vs Stacking
# ============================================================================

from scipy import stats

PARES = [
    ('XGBoost',  'LightGBM'),
    ('XGBoost',  'Stacking'),
    ('LightGBM', 'Stacking'),
]

ALPHA = 0.05
resultados_pares = []

print('═' * 70)
print('TEST DE WILCOXON SIGNED-RANK PAREADO (5 folds, α=0.05)')
print('═' * 70)
print()
print(f'{"Comparación":<28} {"p-AUC":>10} {"p-F1":>10}  Veredicto')
print('─' * 70)

for m1, m2 in PARES:
    stat_auc, p_auc = stats.wilcoxon(aucs[m1], aucs[m2], alternative='two-sided')
    stat_f1,  p_f1  = stats.wilcoxon(f1s[m1],  f1s[m2],  alternative='two-sided')
    
    # Interpretación combinada
    sig_auc = p_auc < ALPHA
    sig_f1  = p_f1  < ALPHA
    
    if sig_auc and sig_f1:
        veredicto = f'❌ {m1} mejor (sig. en AUC y F1)'
    elif sig_auc or sig_f1:
        veredicto = f'⚠️  {m1} mejor solo en una métrica'
    else:
        veredicto = '✅ EMPATE TÉCNICO'
    
    par_str = f'{m1} vs {m2}'
    print(f'{par_str:<28} {p_auc:>10.4f} {p_f1:>10.4f}  {veredicto}')
    
    resultados_pares.append({
        'modelo_1':    m1,
        'modelo_2':    m2,
        'auc_mean_1':  float(np.mean(aucs[m1])),
        'auc_mean_2':  float(np.mean(aucs[m2])),
        'f1_mean_1':   float(np.mean(f1s[m1])),
        'f1_mean_2':   float(np.mean(f1s[m2])),
        'wilcoxon_auc': {'statistic': float(stat_auc), 'p_value': float(p_auc)},
        'wilcoxon_f1':  {'statistic': float(stat_f1),  'p_value': float(p_f1)},
        'veredicto':   veredicto.replace('❌ ', '').replace('⚠️  ', '').replace('✅ ', ''),
    })

print('─' * 70)


══════════════════════════════════════════════════════════════════════
TEST DE WILCOXON SIGNED-RANK PAREADO (5 folds, α=0.05)
══════════════════════════════════════════════════════════════════════

Comparación                       p-AUC       p-F1  Veredicto
──────────────────────────────────────────────────────────────────────
XGBoost vs LightGBM              0.0625     0.1250  ✅ EMPATE TÉCNICO
XGBoost vs Stacking              0.0625     0.6250  ✅ EMPATE TÉCNICO
LightGBM vs Stacking             0.1875     0.8125  ✅ EMPATE TÉCNICO
──────────────────────────────────────────────────────────────────────


In [5]:
# ============================================================================
# CELDA 5: GUARDAR RESULTADO Y GENERAR CITA PARA MEMORIA
# ============================================================================

resultado = {
    'fecha':         pd.Timestamp.now().isoformat(),
    'modelos':       list(modelos_ref.keys()),
    'cv_folds':      5,
    'random_state':  42,
    'n_train':       int(len(y_train)),
    'aucs':          {nombre: [round(x, 6) for x in aucs[nombre]] for nombre in modelos_ref},
    'f1s':           {nombre: [round(x, 6) for x in f1s[nombre]]  for nombre in modelos_ref},
    'mean_std': {
        nombre: {
            'auc_mean': float(np.mean(aucs[nombre])),
            'auc_std':  float(np.std(aucs[nombre])),
            'f1_mean':  float(np.mean(f1s[nombre])),
            'f1_std':   float(np.std(f1s[nombre])),
        }
        for nombre in modelos_ref
    },
    'comparaciones': resultados_pares,
    'alpha':         0.05,
}

ruta_json = RUTA_RESULTS / 'wilcoxon_top3.json'
ruta_json.parent.mkdir(parents=True, exist_ok=True)
with open(ruta_json, 'w', encoding='utf-8') as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False)

print(f'💾 Resultado guardado:')
print(f'   {ruta_json}')
print()

# ── Generar cita para memoria ────────────────────────────────────────────────
n_empates = sum(1 for r in resultados_pares if 'EMPATE' in r['veredicto'].upper())

print('═' * 70)
print('📝 CITA SUGERIDA PARA MEMORIA DEL TFM')
print('═' * 70)
print()
print('   "El test de Wilcoxon signed-rank pareado (n=5 folds, α=0.05) entre')
print('    los 3 mejores modelos del ranking F5 (XGBoost, LightGBM, Stacking)')
print(f'    arroja {n_empates} de 3 comparaciones sin diferencias estadísticamente')
print('    significativas (ver Tabla X). Por ello, la elección final entre estos')
print('    modelos se sustenta en criterios complementarios: estabilidad CV')
print('    (auc_std), interpretabilidad (SHAP/LIME), eficiencia computacional')
print('    y coherencia con la aplicación productiva."')
print()
print('═' * 70)
print('📊 TABLA PARA MEMORIA (Markdown)')
print('═' * 70)
print()
print('| Comparación | p-value AUC | p-value F1 | Veredicto |')
print('|---|---|---|---|')
for r in resultados_pares:
    par = f'{r["modelo_1"]} vs {r["modelo_2"]}'
    print(f'| {par} | {r["wilcoxon_auc"]["p_value"]:.4f} | {r["wilcoxon_f1"]["p_value"]:.4f} | {r["veredicto"]} |')


💾 Resultado guardado:
   c:\PRUEBAS\AU_UJI_v2_RUTA_B\data\05_modelado\results\wilcoxon_top3.json

══════════════════════════════════════════════════════════════════════
📝 CITA SUGERIDA PARA MEMORIA DEL TFM
══════════════════════════════════════════════════════════════════════

   "El test de Wilcoxon signed-rank pareado (n=5 folds, α=0.05) entre
    los 3 mejores modelos del ranking F5 (XGBoost, LightGBM, Stacking)
    arroja 3 de 3 comparaciones sin diferencias estadísticamente
    significativas (ver Tabla X). Por ello, la elección final entre estos
    modelos se sustenta en criterios complementarios: estabilidad CV
    (auc_std), interpretabilidad (SHAP/LIME), eficiencia computacional
    y coherencia con la aplicación productiva."

══════════════════════════════════════════════════════════════════════
📊 TABLA PARA MEMORIA (Markdown)
══════════════════════════════════════════════════════════════════════

| Comparación | p-value AUC | p-value F1 | Veredicto |
|---|---|---|---|
| XGB